In [ ]:
import pandas as pd

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

data = {name: pd.read_csv(path) for name, path in files.items()}

for name, df in data.items():
    print(f"{name:22s} {df.shape[0]:>8,} baris  {df.shape[1]:>2} kolom")

customers                99,441 baris   5 kolom
geolocation            1,000,163 baris   5 kolom
items                   112,650 baris   7 kolom
payments                103,886 baris   5 kolom
reviews                  99,224 baris   7 kolom
orders                   99,441 baris   8 kolom
products                 32,951 baris   9 kolom
sellers                   3,095 baris   4 kolom
category_translation         71 baris   2 kolom


In [ ]:
orders = data["orders"]
print(orders.head(3).T)
print()
print(orders["order_status"].value_counts())
print()
print("Tanggal pesanan paling awal :", orders["order_purchase_timestamp"].min())
print("Tanggal pesanan paling akhir:", orders["order_purchase_timestamp"].max())

                                                              0  \
order_id                       e481f51cbdc54678b7cc49136f2d6af7   
customer_id                    9ef432eb6251297304e76186b10a928d   
order_status                                          delivered   
order_purchase_timestamp                    2017-10-02 10:56:33   
order_approved_at                           2017-10-02 11:07:15   
order_delivered_carrier_date                2017-10-04 19:55:00   
order_delivered_customer_date               2017-10-10 21:25:13   
order_estimated_delivery_date               2017-10-18 00:00:00   

                                                              1  \
order_id                       53cdb2fc8bc7dce0b6741e2150273451   
customer_id                    b0830fb4747a6c6d20dea0b8c802d7ef   
order_status                                          delivered   
order_purchase_timestamp                    2018-07-24 20:41:37   
order_approved_at                           2018-07-26 03:24:

In [ ]:
orders = data["orders"].copy()

# ubah kolom tanggal dari teks menjadi tanggal
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

# satu skor review per pesanan (sebagian pesanan punya lebih dari satu review)
reviews = data["reviews"].groupby("order_id", as_index=False)["review_score"].mean()

# nama kategori produk dalam bahasa Inggris
products = data["products"].merge(
    data["category_translation"], on="product_category_name", how="left"
)[["product_id", "product_category_name_english"]]

print("Tabel siap.")

Tabel siap.


In [ ]:
df = (
    data["items"]
    .merge(orders, on="order_id", how="inner")
    .merge(
        data["customers"][["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
        on="customer_id", how="left",
    )
    .merge(products, on="product_id", how="left")
    .merge(reviews, on="order_id", how="left")
)

print("Baris tabel items awal :", len(data["items"]))
print("Baris setelah digabung :", len(df))
print("Duplikat (order, item) :", df.duplicated(["order_id", "order_item_id"]).sum())

Baris tabel items awal : 112650
Baris setelah digabung : 112650
Duplikat (order, item) : 0


In [ ]:
# kolom bantu untuk analisis
df["order_month"]   = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["delay_days"]    = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
df["is_late"]       = df["delay_days"] > 0

# hanya pesanan yang sudah sampai ke pelanggan
df_clean = df[(df["order_status"] == "delivered") & df["order_delivered_customer_date"].notna()].copy()

print("Baris sebelum dibersihkan:", len(df))
print("Baris setelah dibersihkan:", len(df_clean))
print()
print("Pesanan per bulan:")
print(df_clean.groupby("order_month")["order_id"].nunique().to_string())
print()
print("Persentase terlambat :", round(df_clean["is_late"].mean() * 100, 1), "%")
print("Rata-rata review (tepat waktu):", round(df_clean.loc[~df_clean["is_late"], "review_score"].mean(), 2))
print("Rata-rata review (terlambat)  :", round(df_clean.loc[df_clean["is_late"], "review_score"].mean(), 2))

Baris sebelum dibersihkan: 112650
Baris setelah dibersihkan: 110189

Pesanan per bulan:
order_month
2016-09-01       1
2016-10-01     265
2016-12-01       1
2017-01-01     750
2017-02-01    1653
2017-03-01    2546
2017-04-01    2303
2017-05-01    3545
2017-06-01    3135
2017-07-01    3872
2017-08-01    4193
2017-09-01    4150
2017-10-01    4478
2017-11-01    7288
2017-12-01    5513
2018-01-01    7069
2018-02-01    6555
2018-03-01    7003
2018-04-01    6798
2018-05-01    6749
2018-06-01    6096
2018-07-01    6156
2018-08-01    6351

Persentase terlambat : 6.6 %
Rata-rata review (tepat waktu): 4.21
Rata-rata review (terlambat)  : 2.26


In [ ]:
# periode dengan data lengkap
final = df_clean[(df_clean["order_month"] >= "2017-01-01") &
                 (df_clean["order_month"] <= "2018-08-01")].copy()

# kolom tambahan untuk dashboard
final["total_value"]     = final["price"] + final["freight_value"]
final["delivery_status"] = final["is_late"].map({True: "Late", False: "On time"})
final["order_date"]      = final["order_purchase_timestamp"].dt.strftime("%Y-%m-%d")
final["order_month"]     = final["order_month"].dt.strftime("%Y-%m-%d")

cols = [
    "order_id", "order_item_id", "order_date", "order_month",
    "product_category_name_english", "customer_state", "customer_city",
    "price", "freight_value", "total_value",
    "review_score", "delivery_days", "delay_days", "delivery_status",
]
final = final[cols].rename(columns={"product_category_name_english": "product_category"})
final.to_csv("olist_clean.csv", index=False)

print("Baris :", len(final))
print("Pesanan unik :", final["order_id"].nunique())
print("Kategori kosong :", final["product_category"].isna().sum())
print("Review kosong   :", final["review_score"].isna().sum())

from google.colab import files
files.download("olist_clean.csv")

Baris : 109872
Pesanan unik : 96203
Kategori kosong : 1557
Review kosong   : 823


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>